# Simulations of Park, Wong-Campos et al. (2025): the published NEURON and Izhikevich models

Paper: [doi:10.1038/s41467-025-55819-9](https://doi.org/10.1038/s41467-025-55819-9). The model
claims use the paper's Supplementary Software 1 (41467_2025_55819_MOESM7_ESM.zip, sha256
9eab52ac...7049249, as in the claims file), the NEURON model, and Supplementary Software 2
(41467_2025_55819_MOESM8_ESM.zip, sha256 a5a1b593...0c6b6), the two-compartment Izhikevich model,
both downloaded from the journal and run here. They hold no data from the dandiset (DANDI:001266,
version 0.250710.0758, used by the other notebook of this entry).

Changes needed to run the NEURON model, none of which touches the model:
- NEURON 9 (the only version with wheels for this Python) refuses non-ASCII characters in mod
  files; they occur only in comments and are removed. The paper used NEURON 8.2.
- `run_morphology.py` calls `FullCA1Model(channel_params, morphology_file)`, but the class needs
  a third argument, `ka_cutoff`; the 100 µm that `run_simulation.py` passes is supplied.
- The scripts' plotting is not run; their simulation calls are, with the parameters of
  `main.py`, the two `parameters_list.py` files, Table 1 and the phase-diagram script.
- The morphology (distances, names, main path) is computed once with `run_morphology.py` and shared
  by all runs, and each run goes to a fresh worker process (4 at a time), since the model's per-step
  callback stays registered in NEURON after the model is deleted.

The Izhikevich model is a MATLAB script; no MATLAB or Octave is available here, so it is ported to
Python line by line (the source is printed below for comparison).

| Claim | Tested here |
|---|---|
| fig4f-g-model-accelerometer | tested, with the file's and Table 1's parameters |
| figs20ab-model-patch-vs-opto | tested |
| figs20-model-robustness | tested, the three phase diagrams of Fig. S20c-e (429 runs) |
| figs21-izhikevich | tested, with a Python port of the MATLAB script |

In [ ]:
import hashlib
import importlib.util
import io
import multiprocessing
import os
import re
import subprocess
import sys
import tempfile
import time
import zipfile
from contextlib import redirect_stdout

import matplotlib
import numpy as np
import requests

URL = ("https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-55819-9/MediaObjects/"
       "41467_2025_55819_MOESM7_ESM.zip")
SHA256 = "9eab52ac20d889991fc7ba5f24130d0d80f7d53c3ea69057c5af0f4007049249"
T_START = time.time()
blob = requests.get(URL, timeout=120).content
print("sha256 matches the claims file's prefix and suffix:", hashlib.sha256(blob).hexdigest() == SHA256)
work = tempfile.mkdtemp()
root = "Supplementary NEURON models/Conductance-based model/"
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    phase_script = z.read(root + "Model Robustness/Figures/Figure_S19c-e/Plot_Phase_Diagrams.m").decode()
    for name in z.namelist():
        if not name.startswith(root) or name.endswith("/"):
            continue
        rel = name[len(root):]
        top = rel.split("/")[0]
        if rel.endswith(".py") and "/" not in rel:
            target = os.path.join(work, rel)
        elif rel.startswith("Model Robustness/Channels") and rel.endswith((".mod", ".inc")):
            target = os.path.join(work, os.path.basename(rel))
        elif rel.endswith("parameters_list.py"):
            target = os.path.join(work, "params_" + ("robustness" if top == "Model Robustness" else "oblique") + ".py")
        else:
            continue
        data = z.read(name)
        if target.endswith((".mod", ".inc")):
            data = bytes(c for c in data if c < 128)  # NEURON 9 rejects non-ASCII (comments only)
        with open(target, "wb") as f:
            f.write(data)
nrnivmodl = os.path.join(os.path.dirname(sys.executable), "nrnivmodl")
build = subprocess.run([nrnivmodl], cwd=work, capture_output=True, text=True)
print("nrnivmodl exit code:", build.returncode)
os.chdir(work)
sys.path.insert(0, work)
matplotlib.use("Agg")

import neuron  # noqa: E402  (after the mechanisms are built in the working directory)
import run_morphology as rm  # noqa: E402
from FullCA1Model import FullCA1Model  # noqa: E402
from delete_neuron import delete_all_sections  # noqa: E402
from run_simulation import generate_waveform, run_simulation  # noqa: E402

rm.FullCA1Model = lambda cp, mf: FullCA1Model(cp, mf, 100)  # ka_cutoff as in run_simulation.py

def load_params(name):
    spec = importlib.util.spec_from_file_location(name, os.path.join(work, name + ".py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.setup_parameters

# Channel parameters of main.py
NAV_INACTIVATION = [1, 0.6, 300, 100]
KDR = [0.04, 0.04, 0, 1]
OFF = lambda d, z: [1e-9, 1e-9, d, z]
KEYS = ("duration stim_start stim_end chr_params synapse_params sim_params outputs waveform waveform_patch "
        "patch_params nav kad kap cal car").split()

def get_params(file, run_nr):
    """The parameters of one run of a parameters_list.py, with main.py's NaV slow inactivation."""
    with redirect_stdout(io.StringIO()):
        p = dict(zip(KEYS, load_params(file)(run_nr)))
    p["inact"] = list(NAV_INACTIVATION)
    return p

def channels(p):
    return [p["nav"], p["inact"], p["kap"], p["kad"], KDR, p["chr_params"], OFF(550, 100), OFF(300, 100), p["cal"],
            OFF(50, 30), OFF(300, 100), OFF(550, 100), p["car"], OFF(550, 100), OFF(200, 100),
            OFF(200, 100), OFF(550, 100)]

# The morphology (distances, names, main apical path) does not depend on the channels; it is computed
# once with run_morphology.py and shared by every run.
morpho = rm.run_morphology(channels(get_params("params_robustness", 3)), "Jarsky", "segments")
delete_all_sections()
MORPHO = dict(dist=[[float(x) for x in row] for row in morpho[1]], path=np.array(morpho[3]), names=list(morpho[0]))
MORPHO["D"] = np.array(MORPHO["dist"][0])

def simulate(p):
    """One run of run_simulation.py; returns time, the voltage of every compartment and the morphology."""
    result = run_simulation(p["sim_params"], channels(p), p["synapse_params"], p["waveform"], ["v"], "Jarsky",
                            p["patch_params"], MORPHO["dist"], "segments")
    delete_all_sections()
    t = np.asarray(result[0], dtype=float)
    v = np.array([np.asarray(x, dtype=float) for x in result[1]["v"]])  # = reshape_v of the flattened list
    return dict(t=t, v=v, D=MORPHO["D"], path=MORPHO["path"], names=MORPHO["names"], nav=p["nav"], kad=p["kad"],
                inact=p["inact"], stim=(p["stim_start"], p["stim_end"]), duration=p["duration"])

def quiet(fn, *args):
    with redirect_stdout(io.StringIO()):
        return fn(*args)

def pool_map(fn, jobs, workers=4):
    """Runs fn over jobs in 4 forked processes, a fresh process for each job, so that every run starts from
    the same NEURON state (FullCA1Model registers a per-step CVode callback, update_gmax, that outlives the
    model, so later runs in one process get slower and also run the earlier models' callbacks)."""
    with multiprocessing.get_context("fork").Pool(workers, maxtasksperchild=1) as pool:
        return pool.map(fn, jobs, chunksize=1)

def run_file(job):
    return quiet(simulate, get_params(*job))

t0 = time.time()
names = ["soma step", "wide-field step", "patch, oblique", "optogenetic, oblique"]
jobs = [("params_robustness", 3), ("params_robustness", 12), ("params_oblique", 0), ("params_oblique", 1)]
runs = dict(zip(names, pool_map(run_file, jobs)))
print(f"four runs in {time.time() - t0:.0f} s (4 processes)")

In [ ]:
print("NEURON", neuron.__version__)
for name, r in runs.items():
    print(f"{name}: {r['v'].shape[0]} compartments, {len(r['t'])} time points to {r['t'][-1]:.0f} ms, "
          f"distance 0 to {r['D'].max():.0f} µm, main path {len(r['path'])} compartments, "
          f"v {r['v'].min():.1f} to {r['v'].max():.1f} mV, NaV {r['nav']}, KA {r['kad']}, NaV slow inactivation {r['inact']}")

Membrane potentials stay between about -80 and +50 mV and the morphology spans 0 to about 840 µm
from the soma, as for a CA1 pyramidal cell. The "Model Robustness" parameters give NaV 50 and 48
mS/cm² (soma and distal); Table 1 gives 50 and 47.

## fig4f-g-model-accelerometer

> In the multi-compartment CA1 model, step stimulation at the soma gives a dSpike motif of
> failure, success, success, failures, and wide-field stimulation gives alternating successes
> and failures followed by repeated failures.

Runs "Optopatch step soma" (run 3) and "Optopatch step widefield" (run 12) of `parameters_list.py`
(Model Robustness folder, the only one with these runs). A bAP succeeds when the compartment of the
main apical path nearest 500 µm from the soma peaks above -40 mV before the next somatic spike
(Methods, Parameter fitting; Fig. S20 legend). Somatic spikes are upward crossings of 0 mV. First
the runs as the file gives them (shown as a variant):

In [ ]:
def trace_500(r):
    """Soma and the main-path compartment nearest 500 µm from the soma."""
    D, path = r["D"], r["path"]
    soma = int(np.argmin(D))
    j = int(path[np.argmin(np.abs(D[path] - 500))])
    return dict(t=r["t"], v_soma=r["v"][soma], v_500=r["v"][j], d_500=float(D[j]))

def motif(r):
    if "v_500" not in r:
        r = trace_500(r)
    t, vs, vj = r["t"], r["v_soma"], r["v_500"]
    up = np.flatnonzero((vs[1:] >= 0) & (vs[:-1] < 0)) + 1
    seq, peaks = "", []
    for k, u in enumerate(up):
        end = up[k + 1] if k + 1 < len(up) else len(t)
        pk = vj[u:end].max()
        peaks.append(round(float(pk), 1))
        seq += "S" if pk > -40 else "F"
    return dict(compartment_um=round(r["d_500"], 1), spikes_ms=np.round(t[up], 1).tolist(), peaks_mV=peaks, motif=seq)

print("Model Robustness parameters_list.py as provided (distal NaV 48 mS/cm², slow inactivation ar2 0.6):")
for name in ["soma step", "wide-field step"]:
    m = motif(runs[name])
    print(" ", name, "| stimulus", runs[name]["stim"], "ms |", m)

The criterion asks for Table 1's parameters. Table 1 against the code: NaV 50 (proximal) and 47
(distal) mS/cm², d 300 µm, z 100 µm, where the file has 48 distal; A-type KV -200 and 300, d 0, z
150, as in the file; KDR 40 and 40, as in `main.py`; NaV slow inactivation "s max" 0% proximal and
60% distal, d 300, z 100. In `na3.mod`, `ar2` = 1 means no slow inactivation and 0 maximal;
`main.py` sets [1, 0.6, 300, 100], so its distal value 0.6 matches "60%" if s max is read as the
fraction available, while reading s max as the inactivated fraction gives ar2 0.4. Both are run.
The axonal values and the passive properties are hard-coded in `FullCA1Model.py`; they match Table
1 except the NaV of the plain "axon" sections (70 mS/cm² in the code, 50 in Table 1) and the
specific membrane resistance (40,000 Ω·cm² divided by a spine factor of 2 in the code, 80,000 in
Table 1). Those are code, not parameters, and are left as provided.

The file's step runs last 85 ms (soma, stimulus 25 to 75 ms) and 100 ms (wide-field, 5 to 95 ms).
Neither the Fig. 4f, g legend nor the Methods give the simulated step duration; the experimental
steps were > 200 ms (Results, paragraph on Fig. 4a). A 300 ms version of each run (stimulus to 10
or 5 ms before the end, as in the file) is added.

In [ ]:
def table1(file, run_nr, inact_distal, duration=None):
    """A run of parameters_list.py with Table 1's NaV densities (50 and 47 mS/cm²) and the given distal ar2.

    With a duration, the step is lengthened: the run ends `duration` ms after 0, the stimulus ends as many ms
    before the end as in the file, and the ChR conductance window, waveform and (zero) patch waveform follow."""
    p = get_params(file, run_nr)
    p["nav"] = [0.05, 0.047, 300, 100]
    p["inact"] = [1, inact_distal, 300, 100]
    if duration is not None:
        tail = p["duration"] - p["stim_end"]
        p["duration"], p["stim_end"] = duration, duration - tail
        sp = list(p["sim_params"])
        sp[0] = duration
        p["sim_params"] = sp
        p["waveform"] = generate_waveform(sp, float(np.max(p["waveform"])), p["stim_start"], p["stim_end"], "step")
        p["patch_params"] = [p["patch_params"][0], np.zeros(len(p["waveform"])), p["patch_params"][2]]
        chr_params = [list(c) for c in p["chr_params"]] if isinstance(p["chr_params"][0], list) else [list(p["chr_params"])]
        for c in chr_params:
            c[5] = p["stim_end"]
        p["chr_params"] = chr_params
    return p

def run_table1(job):
    label, file, run_nr, inact_distal, duration = job
    p = table1(file, run_nr, inact_distal, duration)
    t0 = time.time()
    r = quiet(simulate, p)
    out = trace_500(r)
    out.update(label=label, nav=p["nav"], inact=p["inact"], stim=(p["stim_start"], p["stim_end"]),
               duration=p["duration"], seconds=time.time() - t0)
    return out

t1_jobs = []
for ar2 in (0.6, 0.4):  # Table 1 "s max 60%": ar2 = 0.6 as main.py codes it, or 0.4 if 60% is the inactivated fraction
    t1_jobs += [(f"soma step, ar2 {ar2}", "params_robustness", 3, ar2, None),
                (f"wide-field step, ar2 {ar2}", "params_robustness", 12, ar2, None),
                (f"soma step 300 ms, ar2 {ar2}", "params_robustness", 3, ar2, 300),
                (f"wide-field step 300 ms, ar2 {ar2}", "params_robustness", 12, ar2, 300)]
t0 = time.time()
t1_runs = pool_map(run_table1, t1_jobs)
print(f"{len(t1_jobs)} runs in {time.time() - t0:.0f} s (4 processes)")
print("Table 1 parameters (NaV 50 and 47 mS/cm², KA -200 and 300, KDR 40, slow inactivation d 300 z 100):")
for r in t1_runs:
    m = motif(r)
    print(f"  {r['label']}: NaV {r['nav']}, inactivation {r['inact']}, stimulus {r['stim']} ms of {r['duration']:.0f} ms")
    print(f"    motif {m['motif']} ({len(m['motif'])} somatic spikes); bAP peaks at {m['compartment_um']} µm (mV): {m['peaks_mV']}")
    print(f"    somatic spikes (ms): {m['spikes_ms']}")

**Result: not reproduced.** With Table 1's parameters (distal NaV 47 mS/cm², distal ar2 0.6 as in
`main.py`), the soma step gives SSSSF in the file's 85 ms run and SSSSF followed by failures to the
end (21 spikes) in the 300 ms run: the first bAP already peaks at -23.5 mV at 494 µm, so the
opening failure is missing. The wide-field step gives SSS then alternating F and S to the end of the
100 ms run (13 spikes); in the 300 ms run the alternation goes on to the 16th spike, then F, F, S
(the soma stops spiking after 128 ms), so the alternation appears without the opening failure and
without a clear run of closing failures. With ar2 0.4 (s max read as the inactivated fraction) the
soma step gives SSFFF (85 ms) and SS then 19 failures (300 ms), and the wide-field step SSFFFS then
failures, without alternation. The file's own parameters (distal NaV 48) give SSSSF and
SSSFSFSFSFSFS, as before. No run gives the soma motif F, S, S, F.

## figs20ab-model-patch-vs-opto

> In the model, a 2 nA dendritic current injection evokes dendrite-localized sodium spikes,
> while optogenetic stimulation of the same dendrite evokes a somatic spike first that then
> back-propagates.

Runs "Patch pulse oblique" and "Optopatch pulse oblique" of the "Oblique Patch Clamp and
Optogenetic Stimulus" folder (section 98 in both). For each run, the compartment that first
crosses 0 mV ("peaks first" read as "first reaches 0 mV"), the first outside the axon, and the
times at which the soma and the stimulated dendrite (where the current injection first spikes)
first do.

In [ ]:
AXON = ("axon", "node", "inode", "hill", "ais", "myelin")
site = None
for name in ["patch, oblique", "optogenetic, oblique"]:
    r = runs[name]
    t, v, D, names = r["t"], r["v"], r["D"], r["names"]
    cross = np.array([np.argmax(row >= 0) if (row >= 0).any() else len(t) for row in v])
    tc = lambda i: float(t[cross[i]]) if cross[i] < len(t) else np.nan
    dend = np.array([not n.lower().startswith(AXON) for n in names])
    first_any = int(np.argmin(cross))
    first_dend = int(np.flatnonzero(dend)[np.argmin(cross[dend])])
    soma = int(np.argmin(D))
    if site is None:
        site = first_dend  # the stimulated dendrite: where the current injection first spikes
    print(f"{name}: first compartment over 0 mV {names[first_any]} at {tc(first_any):.2f} ms ({D[first_any]:.0f} µm); "
          f"first outside the axon {names[first_dend]} at {tc(first_dend):.2f} ms ({D[first_dend]:.0f} µm); "
          f"soma {names[soma]} at {tc(soma):.2f} ms; stimulated site {names[site]} at {tc(site):.2f} ms; "
          f"compartments over 0 mV: {int((cross < len(t)).sum())} of {len(cross)}")

**Result: reproduced.** The current injection into the oblique dendrite drives that dendrite
(apic[23], 186 µm from the soma) over 0 mV at 15.02 ms while the soma never reaches 0 mV: a
dendrite-localized spike (10 of 1265 compartments over 0 mV). The optogenetic stimulus of the same
dendrite first drives the axon (a node, 25.02 ms) and the initial segment (25.10 ms), then the
soma (25.35 ms) over 0 mV, and 658 compartments follow, while the stimulated dendrite itself does
not reach 0 mV: the spike starts perisomatically, in the axon a fraction of a millisecond before
the soma, and back-propagates.

## figs20-model-robustness

> In the model, the accelerometer motif of bAP failure, dSpikes, then failure persists over a
> wide range of dendritic NaV density, NaV slow inactivation, A-type KV density and stimulus
> strength.

The zip's `Plot_Phase_Diagrams.m` (folder "Figure_S19c-e", the panels published as Fig. S20c-e)
lists the grids and the stimulus of the three diagrams, and holds the published result of each run
as a 6-digit code of the first 6 bAPs (1 = dSpike at 500 µm). Fig. S20c: somatic NaV (11 values)
× dendritic NaV (14), stimulus 0.7 mS/cm²; Fig. S20d: stimulus (11) × dendritic NaV (14), somatic
NaV 50; Fig. S20e: stimulus (11) × non-inactivating distal NaV fraction ar2 (11), NaV 50 and 47.
The script's comment gives the NaV slow inactivation [1, 0.5, 300, 100] for these runs; it is used
for Fig. S20c and d. The script's stimulus labels (15 values of "I_total") do not match its 11
rows; the 11 `stim_max` values are used. 429 runs of 100 ms; on 4 processes this was estimated at
about 32 to 37 min before the notebook was written, and the first 8 runs are timed again below.

Each run is classified with the -40 mV at 500 µm rule (the `motif` function above) on its first 6
bAPs, as the script does. The script's classes are FFFFFF, FSFFFF, FSSFFF, FSSSFF, FSSSSF, FSSSSS,
first-bAP successes (SFFFFF, SSFFFF, ...) and "alternating" (any other code); the motif of the
claim (opening failure, dSpikes, then failure) is FS+F+, that is FSFFFF, FSSFFF, FSSSFF or FSSSSF.
The criterion: the motif cells form one connected region (4-neighbour) in each diagram, and it
includes the Table 1 parameters: NaV 50 and 47 in Fig. S20c; dendritic NaV 47 in Fig. S20d; ar2 0.6
(or 0.4, see above) in Fig. S20e. Table 1 gives no stimulus; the cells at 0.7 mS/cm² (the stimulus
of Fig. S20c) are the ones tested, those at 0.5 mS/cm² (near the 0.49 of run 3) are shown.

In [ ]:
# Grids of Plot_Phase_Diagrams.m (Model Robustness/Figures/Figure_S19c-e; these are Fig. S20c-e of the
# published supplement). Stimulus: the block commented in that script, 100 ms run, step from 5 to 95 ms,
# ChR [1, 0, 40, 10] (d1/2 40 µm, z 10 µm), NaV slow inactivation [1, 0.5, 300, 100], KA [-0.2, 0.3, 0, 150].
NAV_SOMA = [0.04, 0.042, 0.044, 0.046, 0.048, 0.05, 0.052, 0.054, 0.056, 0.058, 0.06]
NAV_DEND = [0.042, 0.043, 0.044, 0.045, 0.046, 0.047, 0.048, 0.049, 0.05, 0.051, 0.052, 0.053, 0.054, 0.055]
STIM = [3e-4, 3.5e-4, 4e-4, 4.5e-4, 5e-4, 5.5e-4, 6e-4, 6.5e-4, 7e-4, 7.5e-4, 8e-4]
AR2 = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
DIAGRAMS = {  # rows, columns, and the parameters of cell (row value, column value): nav soma, nav dend, ar2, stim
    "S20c": (NAV_SOMA, NAV_DEND, lambda a, b: (a, b, 0.5, 7e-4)),
    "S20d": (STIM, NAV_DEND, lambda a, b: (0.05, b, 0.5, a)),
    "S20e": (STIM, AR2, lambda a, b: (0.05, 0.047, b, a)),
}

def sweep_params(nav_soma, nav_dend, ar2, stim):
    p = get_params("params_robustness", 3)  # KA, Ca and the rest as in the file
    duration, s0, s1 = 600 / 6, 50 / 10, 600 / 6 - 50 / 10
    sp = [duration, 0.025, 10, 190, 9, 0]
    waveform = generate_waveform(sp, stim, s0, s1, "step")
    sp[2:6] = [99, 99, 99, 0]
    waveform_patch = generate_waveform(sp, 0, s0, s1, "pulses")
    p.update(duration=duration, stim_start=s0, stim_end=s1, chr_params=[[1, 0, 40, 10, s0, s1, 0, 0]],
             synapse_params=[], sim_params=sp, waveform=waveform, patch_params=[0, waveform_patch, "i"],
             nav=[nav_soma, nav_dend, 300, 100], inact=[1, ar2, 300, 100], kad=[-0.2, 0.3, 0, 150])
    return p

def run_sweep(job):
    key, params = job
    t0 = time.time()
    m = motif(trace_500(quiet(simulate, sweep_params(*params))))
    return key, m["motif"], time.time() - t0

sweep_jobs = [((name, i, j), f(a, b)) for name, (rows, cols, f) in DIAGRAMS.items()
              for i, a in enumerate(rows) for j, b in enumerate(cols)]
print("runs per diagram:", {name: len(rows) * len(cols) for name, (rows, cols, f) in DIAGRAMS.items()},
      "total", len(sweep_jobs))
t0 = time.time()
sweep = pool_map(run_sweep, sweep_jobs[:8])
first8 = time.time() - t0
print(f"first 8 runs on 4 processes: {first8:.0f} s; estimate for the remaining {len(sweep_jobs) - 8}: "
      f"{first8 / 8 * (len(sweep_jobs) - 8) / 60:.0f} min")
t0 = time.time()
sweep += pool_map(run_sweep, sweep_jobs[8:])
print(f"remaining runs: {(time.time() - t0) / 60:.1f} min; mean time of one run inside a worker "
      f"{np.mean([s[2] for s in sweep]):.1f} s")

In [ ]:
codes = {key: seq for key, seq, _ in sweep}
# The published results matrices of Plot_Phase_Diagrams.m, read from the zip (1 = dSpike, first 6 bAPs).
published = []
for block in re.findall(r"\nresults = \[(.*?)\];", phase_script, re.S):
    rows = [r for r in block.split(";") if r.strip()]
    published.append([[tok.strip().zfill(6) for tok in r.replace("\n", " ").split(",") if tok.strip()] for r in rows])

def is_motif(code):
    """Opening failure, then one or more successes, then failures (Plot_Phase_Diagrams.m classes FSFFFF, FSSFFF,
    FSSSFF, FSSSSF); on the first 6 bAPs as in the script."""
    return re.fullmatch("FS+F+", code) is not None

def regions(mask):
    """Number of 4-connected regions of True cells."""
    seen, n = np.zeros_like(mask), 0
    for start in zip(*np.nonzero(mask)):
        if seen[start]:
            continue
        n += 1
        stack = [start]
        while stack:
            i, j = stack.pop()
            if not (0 <= i < mask.shape[0] and 0 <= j < mask.shape[1]) or seen[i, j] or not mask[i, j]:
                continue
            seen[i, j] = True
            stack += [(i + 1, j), (i - 1, j), (i, j + 1), (i, j - 1)]
    return n

TABLE1_CELLS = {"S20c": [(0.05, 0.047)], "S20d": [(7e-4, 0.047), (5e-4, 0.047)],
                "S20e": [(7e-4, 0.6), (5e-4, 0.6), (7e-4, 0.4), (5e-4, 0.4)]}
for k, (name, (rows, cols, f)) in enumerate(DIAGRAMS.items()):
    ours = [[codes[(name, i, j)][:6] for j in range(len(cols))] for i in range(len(rows))]
    pub = [["".join("S" if ch == "1" else "F" for ch in c) for c in row] for row in published[k]]
    mask = np.array([[is_motif(c) for c in row] for row in ours])
    pmask = np.array([[is_motif(c) for c in row] for row in pub])
    print(f"\nFig. {name}: rows {rows}\n  columns {cols}\n  first 6 bAPs at 500 µm (S/F), this run | published matrix")
    for i, a in enumerate(rows):
        print(f"  {a:>7}: " + " ".join(f"{c:<6}" for c in ours[i]) + "  |  " +
              " ".join("M" if m else "." for m in pmask[i]))
    print("  motif map (M = F S+ F+), this run:")
    for i, a in enumerate(rows):
        print(f"  {a:>7}: " + " ".join("M" if m else "." for m in mask[i]))
    same = np.mean([[ours[i][j] == pub[i][j] for j in range(len(cols))] for i in range(len(rows))])
    print(f"  motif cells: {int(mask.sum())} of {mask.size} (published {int(pmask.sum())}); connected regions: "
          f"{regions(mask)} (published {regions(pmask)}); cells with the same 6-bAP pattern as published: {same:.0%}; "
          f"same motif label: {np.mean(mask == pmask):.0%}")
    for a, b in TABLE1_CELLS[name]:
        i, j = rows.index(a), cols.index(b)
        print(f"  cell ({a}, {b}): this run {ours[i][j]} motif={mask[i, j]}; published {pub[i][j]} motif={pmask[i, j]}")

**Result: not reproduced.** All 429 runs of the three diagrams ran (about 34 min on 4
processes). With the -40 mV at 500 µm rule on the first 6 bAPs, the motif F S+ F+ occupies 11 of
154 cells of Fig. S20c, in 2 separate regions at the lowest dendritic NaV (42 to 44 mS/cm²), 17 of
154 cells of Fig. S20d, in one region at dendritic NaV 42 and 43, and none of the 121 cells of Fig.
S20e. In none of the three does the motif region include the Table 1 parameters: the runs there
give SSFFFF (Fig. S20c, NaV 50 and 47; Fig. S20d, 47 at 0.7 mS/cm²), SSSFFF (Fig. S20d at 0.5) and
SSSFFF, SSSSFF or SSFFFF (Fig. S20e, ar2 0.6 or 0.4); the first bAP succeeds there and in most
cells. The published matrices of the same script give the motif in 51, 48 and 88 cells,
including every Table 1 cell (FSFFFF or FSSFFF), and the 6-bAP pattern of this run equals the
published one in only 5%, 5% and 0% of the cells: the provided code, run here, is more excitable in
the distal dendrite than the code that produced the published diagrams. (The published matrices
themselves form 2, 1 and 2 four-connected motif regions.)

## figs21-izhikevich

> A two-compartment Izhikevich model reproduces the opening and closing of the dSpike window
> under soma stimulation and period-doubling under combined soma and dendrite stimulation.

`Two_compartment_Izhikevich.m` of Supplementary Software 2, ported to Python with the same Euler
step, parameters, stimulus and spike rules. Runs: soma only (gs 0.16, gD 0) and combined (gs 0.3,
gD 0.05), each with the Methods' and Fig. S21's somatic reset vrs = -55 and initial dendritic u =
-13 (main reading), and with the script's -65 and -18 (variant). The script defines a dendritic
spike as vD reaching vtD = 0 (then reset to -55 and uD + 1) and a somatic spike as vs reaching 30;
it plots both traces (Fig. S21b, c show the traces), so a somatic spike counts as a dendritic
success (S) when a dendritic spike occurs between it and the next somatic spike, else a failure
(F). The script's plot shows times -5 to 50 (stimulus from 0); the whole 300-unit stimulus is
classified.

In [ ]:
URL2 = ("https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-55819-9/MediaObjects/"
        "41467_2025_55819_MOESM8_ESM.zip")
blob2 = requests.get(URL2, timeout=120).content
sha2 = hashlib.sha256(blob2).hexdigest()
print("sha256", sha2, "| matches the claims file's a5a1b593...0c6b6:", sha2.startswith("a5a1b593") and sha2.endswith("0c6b6"))
with zipfile.ZipFile(io.BytesIO(blob2)) as z:
    m_source = z.read("Supplementary Izhikevich model/Two_compartment_Izhikevich.m").decode()
print(m_source)

In [ ]:
def izhikevich(gs, gd, c_soma, u2_init):
    """Two_compartment_Izhikevich.m, line by line: the same Euler step (dt 0.001, 500000 steps), stimulus
    (off for i < steps/5, on for i < 0.8 steps, off after), equations and spike rules. Returns the times of
    the somatic (v >= 30) and dendritic (v2 >= vmax = 0) spikes on the script's plot axis (i*dt - 100)."""
    a, b, c, d = 0.0025, 0.01, -55, 1
    vmax, rho, couple = 0, 1, 0.325
    v, v2, u, u2 = -75.0, -75.0, -13.0, float(u2_init)
    dt, steps = 0.001, 500000
    soma, dend = [], []
    for i in range(1, steps + 1):
        if i < steps / 5:
            g1 = g2 = 0
        elif i < 0.8 * steps:
            g1, g2 = gs, gd
        else:
            g1 = g2 = 0
        vold, v2old = v, v2
        v = v + dt * (0.04 * v ** 2 + 5 * v + 140 - u - (g1 * v) + (v2old - vold) * couple / rho)
        u = u
        v2 = v2 + dt * (0.04 * v2 ** 2 + 5 * v2 + 140 - u2 - (g2 * v2) - (v2old - vold) * couple)
        u2 = u2 + dt * a * (b * v2 - u2)
        if v >= 30:
            v = c_soma
            soma.append(i * dt - 100)
        if v2 >= vmax:
            v2 = c
            u2 = u2 + d
            dend.append(i * dt - 100)
    return np.array(soma), np.array(dend)

def dendritic_sequence(soma, dend):
    """S when the dendrite spikes (v2 reaches vtD = 0, the script's reset rule) between a somatic spike and the
    next one, F otherwise."""
    nxt = np.append(soma[1:], np.inf)
    return "".join("S" if ((dend >= s) & (dend < n)).any() else "F" for s, n in zip(soma, nxt))

izh = {}
for stim, (gs, gd) in {"soma only": (0.16, 0), "combined": (0.3, 0.05)}.items():
    for reading, (c_soma, u2_init) in {"Methods (vrs -55, uD -13)": (-55, -13),
                                       "script (vrs -65, uD -18)": (-65, -18)}.items():
        soma, dend = izhikevich(gs, gd, c_soma, u2_init)
        seq = dendritic_sequence(soma, dend)
        izh[(stim, reading)] = seq
        shown = int((soma <= 50).sum())  # the script plots -5 to 50
        print(f"{stim}, gs {gs}, gD {gd}, {reading}: {len(soma)} somatic and {len(dend)} dendritic spikes in the "
              f"stimulus (0 to 300)")
        print(f"  per somatic spike: {seq[:shown]} | {seq[shown:shown + 20]}...   ('|' = end of the plotted window, 50)")
        print(f"  first somatic spikes {np.round(soma[:10], 2).tolist()}; dendritic spikes {np.round(dend, 2).tolist()}")

**Result: reproduced** (with a Python port of the MATLAB script). With the Methods' parameters,
soma-only stimulation (gs 0.16) gives failures at the first three somatic spikes, dendritic
spikes at the 4th, 6th and 9th (FFFSFSFFS), then failures at all of the remaining 76 somatic
spikes: the dSpike window opens and closes, though the successes inside it are interleaved with
failures. Combined stimulation (gs 0.3, gD 0.05) gives FSFSFSFSFS (a success at every
other somatic spike, five times), then FFSFFFFFFS and only failures after the 20th somatic spike
(7 dendritic spikes for 154 somatic spikes): period-doubling followed by failure. With the script's own values (vrs -65, uD -18) the soma run
starts with a success (SFSSFS, then failures), so the opening failure is missing in that variant,
and the combined run gives SSSFSSFFSFSFSF then failures.

In [ ]:
print(f"notebook run time: {(time.time() - T_START) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| fig4f-g-model-accelerometer | not reproduced |
| figs20ab-model-patch-vs-opto | reproduced |
| figs20-model-robustness | not reproduced |
| figs21-izhikevich | reproduced (Python port of the MATLAB script) |

The NEURON model as provided, with Table 1's parameters, makes the first bAP succeed at 500 µm in
the step runs and in most cells of the three phase diagrams, while the published phase-diagram
matrices in the same zip show an opening failure there. The difference lies between the published
code and the code that produced the paper's figures (or NEURON 8.2 against 9), not in the rule,
which is the paper's. A failure to reproduce here may be a failure of this attempt, not of the
paper: see the reasoning and the differences from the paper's method.